In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
import pprint

auxip_client, cadip_client, catalog_client, staging_client, prip_client, _ = init_demo()
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

CATALOG_COLLECTION_ID = "SPRINT33_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)

# Stage using a single feature

In [ ]:
items_collection_prip_s2b_l2a_tl = prip_client.search(max_items = 10, collections = ["S2B_L2A_TL"])
single_item = items_collection_prip_s2b_l2a_tl.items[0].to_dict()
assert single_item['type'] == "Feature"
staging_job_id = staging_client.run_staging(single_item, CATALOG_COLLECTION_ID)['prip-station']['jobID']

In [ ]:
assert staging_client.get_job_info(staging_job_id)['status'] == 'successful'

# Stage using link

In [ ]:
items_collection_prip_s2b_l2a_tl = prip_client.search(max_items = 10, collections = ["S2B_L2A_TL"])
single_item_link = next((link.href for link in items_collection_prip_s2b_l2a_tl.items[0].links if link.rel == 'self'), None)
assert 'http://rs-server-prip:8000/prip' in single_item_link
staging_link_job_id = staging_client.run_staging(single_item_link, CATALOG_COLLECTION_ID)['rs-server-prip']['jobID']

In [ ]:
assert staging_client.get_job_info(staging_link_job_id)['status'] == 'successful'